# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Kogut w rosole
2. Maleńkie Królestwo Królewny Aurelki
3. Z miłością na Ty
4. Mistrzowie musicalu | Małgorzata Chruściel i Maciej Pawlak
5. Świadkowie, czyli nasza mała stabilizacja
6. Sławek Uniatowski: The Best Of II
7. Miesiąc Fotografii 2026
8. 19. Festiwal Muzyki Filmowej w Krakowie
9. Juwenalia Krakoskie 2026
10. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”

Czas wykonania: 3.16s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Kogut w rosole
2. Maleńkie Królestwo Królewny Aurelki
3. Z miłością na Ty
4. Mistrzowie musicalu | Małgorzata Chruściel i Maciej Pawlak
5. Świadkowie, czyli nasza mała stabilizacja
6. Sławek Uniatowski: The Best Of II
7. Miesiąc Fotografii 2026
8. 19. Festiwal Muzyki Filmowej w Krakowie
9. Juwenalia Krakoskie 2026
10. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”

Czas wykonania (wątki): 1.71s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 4 procesach (rdzeniach)...
Zakończono w czasie 4.24s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def fetch_cat_fact():
    
    try:
        response = requests.get(CAT_API_URL)
        return response.json().get('fact')
    except Exception as e:
        return f"Błąd: {e}"

def fetch_facts_sequential(num_facts=20):
    
    print(f"\n{'='*60}")
    print("METODA SEKWENCYJNA")
    print(f"{'='*60}")
    
    start_time = time.time()
    facts = []
    
    for i in range(num_facts):
        fact = fetch_cat_fact()
        facts.append(fact)
        print(f"{i+1}. {fact}")
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    print(f"\nCzas wykonania (sekwencyjnie): {elapsed_time:.2f} sekund")
    return facts, elapsed_time

def fetch_facts_threaded(num_facts=20, max_workers=10):
  
    print(f"\n{'='*60}")
    print("METODA WIELOWĄTKOWA")
    print(f"{'='*60}")
    
    start_time = time.time()
    facts = []
    
   
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    
        future_to_index = {executor.submit(fetch_cat_fact): i for i in range(num_facts)}
        
        
        for future in concurrent.futures.as_completed(future_to_index):
            index = future_to_index[future]
            try:
                fact = future.result()
                facts.append(fact)
                print(f"{index+1}. {fact}")
            except Exception as e:
                print(f"{index+1}. Błąd: {e}")
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    print(f"\nCzas wykonania (wielowątkowo): {elapsed_time:.2f} sekund")
    return facts, elapsed_time

def main():
    """Główna funkcja porównująca obie metody"""
    print("\n" + "="*60)
    print("PORÓWNANIE METOD POBIERANIA FAKTÓW O KOTACH")
    print("="*60)
    
    num_facts = 20
    
    # 1. Metoda sekwencyjna
    _, time_sequential = fetch_facts_sequential(num_facts)
    
    # 2. Metoda wielowątkowa
    _, time_threaded = fetch_facts_threaded(num_facts, max_workers=10)
    
    # 3. Porównanie wyników
    print(f"\n{'='*60}")
    print("PODSUMOWANIE")
    print(f"{'='*60}")
    print(f"Liczba faktów:              {num_facts}")
    print(f"Czas sekwencyjny:           {time_sequential:.2f} s")
    print(f"Czas wielowątkowy:          {time_threaded:.2f} s")
    print(f"Przyspieszenie:             {time_sequential/time_threaded:.2f}x")
    print(f"Zaoszczędzony czas:         {time_sequential - time_threaded:.2f} s")
    print(f"Wydajność:                  {((time_sequential - time_threaded) / time_sequential * 100):.1f}% szybciej")
    print(f"{'='*60}\n")

if __name__ == "__main__":
    main()


PORÓWNANIE METOD POBIERANIA FAKTÓW O KOTACH

METODA SEKWENCYJNA
1. During the time of the Spanish Inquisition, Pope Innocent VIII condemned cats as evil and thousands of cats were burned. Unfortunately, the widespread killing of cats led to an explosion of the rat population, which exacerbated the effects of the Black Death.
2. The name "jaguar" comes from a Native American word meaning "he who kills with one leap".
3. The first cat show was in 1871 at the Crystal Palace in London.
4. Some common houseplants poisonous to cats include: English Ivy, iris, mistletoe, philodendron, and yew.
5. A cat uses its whiskers for measuring distances.  The whiskers of a cat are capable of registering very small changes in air pressure.
6. A female cat will be pregnant for approximately 9 weeks or between 62 and 65 days from conception to delivery.
7. Cats dislike citrus scent.
8. A cat will tremble or shiver when it is extreme pain.
9. Cats have 3 eyelids.
10. A female cat will be pregnant for appr

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [7]:
import queue
import threading
import time
import random


shared_queue = queue.Queue()


production_finished = threading.Event()


stats_lock = threading.Lock()
stats = {
    'produced': 0,
    'consumed_even': 0,
    'consumed_odd': 0
}

def producer(num_items=50, delay=0.1):
  
    print(f"PRODUCENT: Rozpoczynam produkcję {num_items} liczb...\n")
    
    for i in range(1, num_items + 1):
        shared_queue.put(i)
        
        with stats_lock:
            stats['produced'] += 1
        
        print(f"Producent: Wyprodukowano liczbę {i} | Kolejka: {shared_queue.qsize()}")
        time.sleep(delay)  
  
    production_finished.set()
    print(f"\nPRODUCENT: Zakończono produkcję! Wyprodukowano {num_items} liczb.\n")

def consumer_even(consumer_id=1):
   
    print(f"KONSUMENT {consumer_id} (PARZYSTE): Gotowy do pracy!\n")
    
    while True:
        try:
            
            number = shared_queue.get(timeout=1)
            
           
            if number % 2 == 0:
                print(f"Konsument {consumer_id}: Przetwarzam liczbę PARZYSTĄ: {number}")
                time.sleep(random.uniform(0.05, 0.15))  
                
                with stats_lock:
                    stats['consumed_even'] += 1
                
                shared_queue.task_done()
            else:
               
                shared_queue.put(number)
                shared_queue.task_done()
        
        except queue.Empty:
            
            if production_finished.is_set() and shared_queue.empty():
                print(f"\nKONSUMENT {consumer_id}: Kończę pracę (brak więcej liczb parzystych).\n")
                break

def consumer_odd(consumer_id=2):
 
    print(f"KONSUMENT {consumer_id} (NIEPARZYSTE): Gotowy do pracy!\n")
    
    while True:
        try:
           
            number = shared_queue.get(timeout=1)
            
      
            if number % 2 != 0:
                print(f"Konsument {consumer_id}: Przetwarzam liczbę NIEPARZYSTĄ: {number}")
                time.sleep(random.uniform(0.05, 0.15))  
                
                with stats_lock:
                    stats['consumed_odd'] += 1
                
                shared_queue.task_done()
            else:
                
                shared_queue.put(number)
                shared_queue.task_done()
        
        except queue.Empty:
      
            if production_finished.is_set() and shared_queue.empty():
                print(f"\nKONSUMENT {consumer_id}: Kończę pracę (brak więcej liczb nieparzystych).\n")
                break

def main():
   
    print("="*70)
    print("SYSTEM PRODUCENT-KONSUMENT: Parzyste vs Nieparzyste")
    print("="*70 + "\n")
    
    NUM_ITEMS = 50  
    
    start_time = time.time()
    
    
    producer_thread = threading.Thread(
        target=producer, 
        args=(NUM_ITEMS, 0.1),
        name="Producent"
    )
    
    consumer1_thread = threading.Thread(
        target=consumer_even,
        args=(1,),
        name="Konsument-Parzyste"
    )
    
    consumer2_thread = threading.Thread(
        target=consumer_odd,
        args=(2,),
        name="Konsument-Nieparzyste"
    )
    
 
    producer_thread.start()
    consumer1_thread.start()
    consumer2_thread.start()
    

    producer_thread.join()
    consumer1_thread.join()
    consumer2_thread.join()
    
    end_time = time.time()
    
    
    print("="*70)
    print("PODSUMOWANIE")
    print("="*70)
    print(f"Liczb wyprodukowanych:        {stats['produced']}")
    print(f"Liczb parzystych przetworzonych:    {stats['consumed_even']}")
    print(f"Liczb nieparzystych przetworzonych: {stats['consumed_odd']}")
    print(f"Razem przetworzonych:         {stats['consumed_even'] + stats['consumed_odd']}")
    print(f"Czas całkowity:               {end_time - start_time:.2f} s")
    print(f"Kolejka końcowa:              {shared_queue.qsize()} elementów")
    print("="*70 + "\n")
    
    
    if stats['consumed_even'] + stats['consumed_odd'] == stats['produced']:
        print("Wszystkie liczby zostały przetworzone poprawnie!")
    else:
        print("Uwaga: Niezgodność w liczbie przetworzonych elementów!")

if __name__ == "__main__":
    main()

SYSTEM PRODUCENT-KONSUMENT: Parzyste vs Nieparzyste

PRODUCENT: Rozpoczynam produkcję 50 liczb...

Producent: Wyprodukowano liczbę 1 | Kolejka: 1
KONSUMENT 1 (PARZYSTE): Gotowy do pracy!

KONSUMENT 2 (NIEPARZYSTE): Gotowy do pracy!

Konsument 2: Przetwarzam liczbę NIEPARZYSTĄ: 1
Producent: Wyprodukowano liczbę 2 | Kolejka: 1
Konsument 1: Przetwarzam liczbę PARZYSTĄ: 2
Producent: Wyprodukowano liczbę 3 | Kolejka: 1Konsument 2: Przetwarzam liczbę NIEPARZYSTĄ: 3

Producent: Wyprodukowano liczbę 4 | Kolejka: 1
Konsument 1: Przetwarzam liczbę PARZYSTĄ: 4
Producent: Wyprodukowano liczbę 5 | Kolejka: 1Konsument 2: Przetwarzam liczbę NIEPARZYSTĄ: 5

Producent: Wyprodukowano liczbę 6 | Kolejka: 1
Konsument 1: Przetwarzam liczbę PARZYSTĄ: 6
Producent: Wyprodukowano liczbę 7 | Kolejka: 1Konsument 2: Przetwarzam liczbę NIEPARZYSTĄ: 7

Producent: Wyprodukowano liczbę 8 | Kolejka: 1
Konsument 1: Przetwarzam liczbę PARZYSTĄ: 8
Producent: Wyprodukowano liczbę 9 | Kolejka: 1Konsument 2: Przetwarzam lic

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [13]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def calculate_sequential(numbers):
   
    print(f"\n{'='*70}")
    print("METODA SEKWENCYJNA")
    print(f"{'='*70}")
    
    start_time = time.time()
    results = []
    
    total = len(numbers)
    for i, n in enumerate(numbers, 1):
        result = calculate_power_sum(n)
        results.append(result)
        
      
        if i % (total // 10) == 0 or i == total:
            progress = (i / total) * 100
            print(f"Postęp: {progress:.0f}% ({i}/{total})")
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    print(f"\nCzas wykonania (sekwencyjnie): {elapsed_time:.2f} sekund")
    return results, elapsed_time

def calculate_parallel(numbers, num_processes=None):
   
    if num_processes is None:
        num_processes = multiprocessing.cpu_count()
    
    print(f"\n{'='*70}")
    print("METODA RÓWNOLEGŁA (MULTIPROCESSING)")
    print(f"{'='*70}")
    print(f"Liczba procesów: {num_processes}")
    
    start_time = time.time()
    
  
    with multiprocessing.Pool(processes=num_processes) as pool:
        
        results = []
        total = len(numbers)
        
     
        for i, result in enumerate(pool.imap(calculate_power_sum, numbers, chunksize=100), 1):
            results.append(result)
            
          
            if i % (total // 10) == 0 or i == total:
                progress = (i / total) * 100
                print(f"Postęp: {progress:.0f}% ({i}/{total})")
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    print(f"\nCzas wykonania (równolegle): {elapsed_time:.2f} sekund")
    return results, elapsed_time

def main():
    """
    Główna funkcja porównująca obie metody
    """
    print("\n" + "="*70)
    print("PORÓWNANIE OBLICZEŃ: Sekwencyjne vs Równoległe")
    print("="*70)
    
   
    RANGE_START = 1
    RANGE_END = 10000
    numbers = list(range(RANGE_START, RANGE_END + 1))
    
    print(f"\nZakres liczb: {RANGE_START} - {RANGE_END}")
    print(f"Liczba obliczeń: {len(numbers)}")
    print(f"Dostępne rdzenie CPU: {multiprocessing.cpu_count()}")
    
    # 1. Metoda sekwencyjna
    results_seq, time_sequential = calculate_sequential(numbers)
    
    # 2. Metoda równoległa
    results_par, time_parallel = calculate_parallel(numbers)
    
    # 3. Porównanie wyników
    print(f"\n{'='*70}")
    print("PODSUMOWANIE")
    print(f"{'='*70}")
    print(f"Zakres:                     {RANGE_START} - {RANGE_END}")
    print(f"Liczba obliczeń:            {len(numbers)}")
    print(f"Użyte procesy:              {multiprocessing.cpu_count()}")
    print(f"\nCzas sekwencyjny:           {time_sequential:.2f} s")
    print(f"Czas równoległy:            {time_parallel:.2f} s")
    print(f"\nPrzyspieszenie:             {time_sequential/time_parallel:.2f}x")
    print(f"Zaoszczędzony czas:         {time_sequential - time_parallel:.2f} s")
    print(f"Wydajność:                  {((time_sequential - time_parallel) / time_sequential * 100):.1f}% szybciej")
  
    if results_seq == results_par:
        print(f"\nWyniki są identyczne - obliczenia poprawne!")
    else:
        print(f"\nUwaga: Wyniki się różnią!")
    
 

if __name__ == "__main__":
    main()


PORÓWNANIE OBLICZEŃ: Sekwencyjne vs Równoległe

Zakres liczb: 1 - 10000
Liczba obliczeń: 10000
Dostępne rdzenie CPU: 4

METODA SEKWENCYJNA
Postęp: 10% (1000/10000)
Postęp: 20% (2000/10000)
Postęp: 30% (3000/10000)
Postęp: 40% (4000/10000)
Postęp: 50% (5000/10000)
Postęp: 60% (6000/10000)
Postęp: 70% (7000/10000)
Postęp: 80% (8000/10000)
Postęp: 90% (9000/10000)
Postęp: 100% (10000/10000)

Czas wykonania (sekwencyjnie): 0.99 sekund

METODA RÓWNOLEGŁA (MULTIPROCESSING)
Liczba procesów: 4
Postęp: 10% (1000/10000)
Postęp: 20% (2000/10000)
Postęp: 30% (3000/10000)
Postęp: 40% (4000/10000)
Postęp: 50% (5000/10000)
Postęp: 60% (6000/10000)
Postęp: 70% (7000/10000)
Postęp: 80% (8000/10000)
Postęp: 90% (9000/10000)
Postęp: 100% (10000/10000)

Czas wykonania (równolegle): 0.85 sekund

PODSUMOWANIE
Zakres:                     1 - 10000
Liczba obliczeń:            10000
Użyte procesy:              4

Czas sekwencyjny:           0.99 s
Czas równoległy:            0.85 s

Przyspieszenie:           